# Script 2 — Preparação & Engenharia de Features (V6 — DFP + ITR + Macro)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Esta versão incorpora **DFPs e ITRs** como observações de treino e enriquece
o dataset com **variáveis macroeconômicas** (BCB/SGS) e **dados de mercado** (yfinance), como observações de treino, aumentando o dataset
de ~74 para ~369 observações — ganho de 5× sem alterar o Script 1.


## Etapa 0 — Dependências, logging e configuração global

In [1]:
import logging, json, pickle, warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)

logger = logging.getLogger('pipeline_preparacao_v6')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
_sh  = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh  = logging.FileHandler(PASTA_SAIDA / 'pipeline_preparacao_v6.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros configuráveis ──────────────────────────────────────────────
LISTA_KPIS = [
    'margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
    'roe','roa','liquidez_corrente','liquidez_imediata',
    'endividamento','alavancagem_de','div_liquida','cobertura_juros',
    'giro_ativo','fco_receita','fco_lucro','EBITDA',
    'FCF','margem_fcf','conversao_caixa',
]
TARGET_COLS = {
    'TARGET_DRE_3.01': 'DRE_3.01',
    'TARGET_DRE_3.11': 'DRE_3.11',
    'TARGET_EBITDA'  : 'EBITDA',
}
LIMIAR_NULO    = 0.80
FATOR_WINSOR   = 3.0
MAX_COLS_YOY   = 16
CLIP_YOY       = 5.0
CORR_MIN       = 0.10
N_FEATURES_RFE = 15
FRAC_TREINO    = 0.75
GAP_YOY_MIN    = 340
GAP_YOY_MAX    = 395

# Período de coleta macro: 2015-2025 completos
ANO_INICIO_MACRO = 2015
ANO_FIM_MACRO    = 2025

logger.info("Script 2 V6 (DFP+ITR) iniciado | pandas=%s", pd.__version__)

COLS_MACRO_FINAL = []  # preenchido pela Etapa 1B


2026-04-29 21:47:24 | INFO     | Script 2 V6 (DFP+ITR) iniciado | pandas=3.0.1


## Etapa 1 — Carregamento e validação

In [2]:
cam_parquet = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
if not cam_parquet.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {cam_parquet}\n"
        "Execute o Script 1 (01_cvm_processamento_V5.ipynb) antes de continuar."
    )

dataset = pd.read_parquet(cam_parquet)
logger.info("Dataset carregado: %d × %d", *dataset.shape)

dataset['DT_REFER']  = pd.to_datetime(dataset['DT_REFER'], errors='coerce', utc=False)
TZ_DATASET           = dataset['DT_REFER'].dt.tz
dataset['ANO']       = dataset['DT_REFER'].dt.year.astype('Int64')
dataset['TRIMESTRE'] = dataset['DT_REFER'].dt.quarter.astype('Int64')
dataset['MES']       = dataset['DT_REFER'].dt.month.astype('Int64')

COLS_OBR = ['CNPJ_CIA','NOME_CIA','SETOR','ANO','ORIGEM','DT_REFER']
faltando = [c for c in COLS_OBR if c not in dataset.columns]
if faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltando}")

n_emp = dataset['NOME_CIA'].nunique()
kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
kpis_ausentes  = [k for k in LISTA_KPIS if k not in dataset.columns]
if kpis_ausentes:
    logger.warning("KPIs ausentes: %s", kpis_ausentes)

for orig in ['DFP','ITR']:
    sub = dataset[dataset['ORIGEM']==orig]
    logger.info("%-3s: %d obs | %d empresas | anos %s",
                orig, len(sub), sub['NOME_CIA'].nunique(),
                sorted(sub['ANO'].dropna().astype(int).unique()))

print(f"\n{'='*60}")
print(f"  Dataset carregado | {dataset.shape[0]:,} × {dataset.shape[1]}")
print(f"  Empresas : {n_emp}/25 | DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()}")
print(f"  KPIs     : {len(kpis_presentes)}/{len(LISTA_KPIS)} | TZ: {TZ_DATASET}")
print(f"{'='*60}")


2026-04-29 21:47:31 | INFO     | Dataset carregado: 983 × 771
2026-04-29 21:47:31 | INFO     | DFP: 230 obs | 25 empresas | anos [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
2026-04-29 21:47:31 | INFO     | ITR: 753 obs | 25 empresas | anos [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]



  Dataset carregado | 983 × 771
  Empresas : 25/25 | DFP: 230 | ITR: 753
  KPIs     : 19/19 | TZ: America/Sao_Paulo


## Etapa 1B — Enriquecimento com Dados Macroeconômicos e de Mercado

Integra variáveis de contexto externo ao dataset financeiro. Duas fontes:

**BCB/SGS:** SELIC (432), IPCA (433), Câmbio (1), PIB trimestral (4380).
Coleta **ano a ano** (2015–2025), 10 requisições por série — evita rejeição
por payload excessivo e delimita o intervalo ao período do estudo.

**Yahoo Finance via yfinance:** volatilidade 60d do EWZ como proxy de risco-Brasil.

**Alinhamento temporal:** `merge_asof` **individual por série** (direction='backward').
Cada série é mergeada separadamente no `df_key` — elimina o bug de cobertura 0%
causado por misturar séries de frequência diária (EWZ) com mensal (BCB) em um
único `pd.DataFrame`, onde o índice dominante (diário) deixava as séries mensais
com NaN em todas as linhas do merge.

**Fallback:** API indisponível → coluna preenchida com NaN. Pipeline não falha.

In [3]:
import requests, warnings
from datetime import datetime

# ── Tickers B3 por empresa ────────────────────────────────────────────────
TICKERS_B3 = {
    'Petrobras': 'PETR4.SA', 'Prio': 'PRIO3.SA', 'Ultrapar': 'UGPA3.SA',
    'Raizen': 'RAIZ4.SA', 'Vibra Energia': 'VBBR3.SA',
    'Engie Brasil': 'EGIE3.SA', 'Equatorial Energia': 'EQTL3.SA',
    'Taesa': 'TAEE11.SA', 'CPFL Energia': 'CPFE3.SA', 'ISA CTEEP': 'TRPL4.SA',
    'Lojas Renner': 'LREN3.SA', 'Magazine Luiza': 'MGLU3.SA',
    'Alpargatas': 'ALPA4.SA', 'Arezzo': 'ARZZ3.SA', 'Grupo Mateus': 'GMAT3.SA',
    'Vale': 'VALE3.SA', 'Suzano': 'SUZB3.SA', 'Klabin': 'KLBN11.SA',
    'Gerdau': 'GGBR4.SA', 'CSN Mineracao': 'CMIN3.SA',
    'WEG': 'WEGE3.SA', 'Totvs': 'TOTS3.SA', 'Positivo': 'POSI3.SA',
    'Intelbras': 'INTB3.SA', 'Brisanet': 'BRIT3.SA',
}

SERIES_BCB = {
    'macro_selic':   432,
    'macro_ipca':    433,
    'macro_cambio':  1,
    'macro_pib_tri': 4380,
}


def buscar_serie_bcb_anual(codigo, ano_inicio=ANO_INICIO_MACRO, ano_fim=ANO_FIM_MACRO):
    """
    Coleta a série BCB ano a ano (uma requisição por ano) cobrindo
    [01/01/ano_inicio .. 31/12/ano_fim].

    Estratégia:
    - Loop de ANO_INICIO_MACRO até ANO_FIM_MACRO inclusive
    - dataInicial = 01/01/ANO  |  dataFinal = 31/12/ANO
    - Evita rejeição por payload excessivo (406/413) e delimita
      o intervalo exatamente ao período do TCC
    - Accept: application/json previne 406 Not Acceptable
    - Fragmentos de cada ano são concatenados no final
    """
    fragmentos = []
    for ano in range(ano_inicio, ano_fim + 1):
        data_ini = f'01/01/{ano}'
        data_fim = f'31/12/{ano}'
        url = (
            f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados'
            f'?formato=json&dataInicial={data_ini}&dataFinal={data_fim}'
        )
        try:
            r = requests.get(url, timeout=20,
                             headers={'Accept': 'application/json'})
            r.raise_for_status()
            dados = r.json()
            if not dados:
                logger.debug('BCB %d | %d: sem registros', codigo, ano)
                continue
            df_ano = pd.DataFrame(dados)
            df_ano['data']  = pd.to_datetime(df_ano['data'], format='%d/%m/%Y')
            df_ano['valor'] = pd.to_numeric(df_ano['valor'], errors='coerce')
            fragmentos.append(df_ano)
            logger.debug('BCB %d | %d: %d obs', codigo, ano, len(df_ano))
        except Exception as e:
            logger.warning('BCB serie %d | ano %d indisponivel: %s', codigo, ano, e)

    if not fragmentos:
        return pd.Series(dtype=float)

    df_total = pd.concat(fragmentos, ignore_index=True)
    df_total = df_total.drop_duplicates('data').sort_values('data')
    return df_total.set_index('data')['valor']


def buscar_dados_mercado(ticker, data_inicio='2015-01-01', data_fim='2025-12-31'):
    """Retorna DataFrame com retorno_12m e volatilidade_60d para o ticker."""
    try:
        import yfinance as yf
        hist = yf.download(ticker, start=data_inicio, end=data_fim,
                           progress=False, auto_adjust=True)
        if hist.empty:
            return pd.DataFrame()
        if isinstance(hist.columns, pd.MultiIndex):
            hist.columns = hist.columns.droplevel(1)
        close   = hist['Close'].squeeze()
        ret_12m = close.pct_change(252).rename('retorno_12m')
        vol_60d = close.pct_change().rolling(60).std().rename('volatilidade_60d')
        return pd.concat([ret_12m, vol_60d], axis=1)
    except Exception as e:
        logger.warning('yfinance %s indisponivel: %s', ticker, e)
        return pd.DataFrame()


# ── Coleta das séries BCB (loop ano a ano) ────────────────────────────────
logger.info('Coletando series macro do BCB/SGS (%d–%d, ano a ano)...',
            ANO_INICIO_MACRO, ANO_FIM_MACRO)
series_bcb = {}
for nome_col, codigo in SERIES_BCB.items():
    s = buscar_serie_bcb_anual(codigo)
    if not s.empty:
        series_bcb[nome_col] = s
        logger.info('  %s: %d obs (%s a %s)',
                    nome_col, len(s),
                    s.index.min().date(), s.index.max().date())
    else:
        logger.warning('  %s: sem dados (fallback para NaN)', nome_col)

# ── CDS Brasil via yfinance ───────────────────────────────────────────────
logger.info('Coletando CDS Brasil (proxy EWZ)...')
df_ewz = buscar_dados_mercado('EWZ')
if not df_ewz.empty and 'volatilidade_60d' in df_ewz.columns:
    vol_ewz = df_ewz['volatilidade_60d'].copy()
    # Normalizar tz para naive antes de entrar no dict
    if vol_ewz.index.tz is not None:
        vol_ewz.index = vol_ewz.index.tz_convert(None)
    series_bcb['macro_vol_brasil'] = vol_ewz.rename('macro_vol_brasil')

# ── Alinhamento temporal — merge_asof INDIVIDUAL por série ───────────────
#
# CORREÇÃO CENTRAL:
# A abordagem anterior criava pd.DataFrame(series_bcb) misturando séries de
# frequências diferentes (BCB mensal + EWZ diário). O índice resultante era
# dominado pelas ~3000 datas diárias do EWZ; nas linhas do merge_asof com
# DT_MACRO diário, macro_ipca e macro_pib_tri apareciam como NaN (cobertura 0%).
#
# Solução: cada série é mergeada individualmente no df_key e o resultado é
# atribuído diretamente como nova coluna do dataset, sem interferência cruzada.
#
if series_bcb:
    # Vetor de datas de referência sem timezone (naive) para o merge
    dt_naive = dataset['DT_REFER'].dt.tz_localize(None)
    df_key   = pd.DataFrame({
        'DT_REFER_naive': dt_naive.values,
        'idx_orig':       dataset.index.tolist(),
    }).sort_values('DT_REFER_naive').reset_index(drop=True)

    cols_macro = []
    for col_name, serie in series_bcb.items():
        # Normalizar tz para naive
        s_naive = serie.copy()
        if s_naive.index.tz is not None:
            s_naive.index = s_naive.index.tz_convert(None)
        else:
            s_naive.index = pd.to_datetime(s_naive.index).tz_localize(None)

        # Construção do DataFrame à prova de versão de pandas:
        # Evita depender de reset_index() + rename, cujo comportamento com
        # index.name=None varia entre pandas 3.0.1 e 3.0.2 (coluna 0 vs 'index'),
        # causando KeyError ao tentar acessar merged[col_name] depois do merge_asof.
        # cast para datetime64[us] garante compatibilidade com df_key independente
        # da resolução retornada pelo BCB (pandas 3.0.1 retorna [s], 3.0.2 retorna [us])
        dt_macro = s_naive.index.astype('datetime64[us]')
        df_s = pd.DataFrame({'DT_MACRO': dt_macro, col_name: s_naive.values})
        df_s = df_s.sort_values('DT_MACRO').reset_index(drop=True)

        # Tolerância: 120 dias cobre PIB trimestral (~90 d de defasagem publicação)
        merged = pd.merge_asof(
            df_key[['DT_REFER_naive', 'idx_orig']],
            df_s,
            left_on='DT_REFER_naive',
            right_on='DT_MACRO',
            direction='backward',
            tolerance=pd.Timedelta('120 days'),
        ).set_index('idx_orig').sort_index()

        # Atribuir pelo índice (não por posição) — alinhamento seguro
        dataset[col_name] = merged[col_name]
        cob = dataset[col_name].notna().mean()
        logger.info('  Macro %-25s integrada | cobertura %.0f%%', col_name, cob * 100)
        cols_macro.append(col_name)

    logger.info('Macro integrada: %d colunas | cobertura media %.0f%%',
                len(cols_macro),
                dataset[cols_macro].notna().mean().mean() * 100 if cols_macro else 0)
else:
    cols_macro = []
    logger.warning('Nenhuma serie macro disponivel — pipeline continua sem dados macro')

COLS_MACRO_FINAL = cols_macro.copy()
print(f'Macro integrada: {len(cols_macro)} colunas')
if cols_macro:
    print(dataset[cols_macro].describe().round(4).to_string())


2026-04-29 21:47:34 | INFO     | Coletando series macro do BCB/SGS (2015–2025, ano a ano)...
2026-04-29 21:47:50 | INFO     |   macro_selic: 4018 obs (2015-01-01 a 2025-12-31)
2026-04-29 21:47:58 | INFO     |   macro_ipca: 132 obs (2015-01-01 a 2025-12-01)
2026-04-29 21:48:10 | INFO     |   macro_cambio: 2760 obs (2015-01-02 a 2025-12-31)
2026-04-29 21:48:17 | INFO     |   macro_pib_tri: 132 obs (2015-01-01 a 2025-12-01)
2026-04-29 21:48:17 | INFO     | Coletando CDS Brasil (proxy EWZ)...
2026-04-29 21:48:18 | INFO     |   Macro macro_selic               integrada | cobertura 100%
2026-04-29 21:48:18 | INFO     |   Macro macro_ipca                integrada | cobertura 100%
2026-04-29 21:48:18 | INFO     |   Macro macro_cambio              integrada | cobertura 100%
2026-04-29 21:48:18 | INFO     |   Macro macro_pib_tri             integrada | cobertura 100%
2026-04-29 21:48:18 | INFO     |   Macro macro_vol_brasil          integrada | cobertura 100%
2026-04-29 21:48:18 | INFO     | Mac

Macro integrada: 5 colunas
       macro_selic  macro_ipca  macro_cambio  macro_pib_tri  macro_vol_brasil
count     983.0000    983.0000      983.0000       983.0000          983.0000
mean        9.8113      0.5067        4.5787   736,713.6339            0.0202
std         4.2025      0.4362        0.9262   185,059.7311            0.0088
min         2.0000     -0.2900        3.1026   490,621.4000            0.0107
25%         6.5000      0.2100        3.8322   578,118.2000            0.0157
50%        10.7500      0.4800        4.9962   705,408.8000            0.0182
75%        13.7500      0.7300        5.4066   905,299.2000            0.0217
max        15.0000      1.6200        6.1923 1,074,994.6000            0.0644


In [5]:
# ── Dados de mercado por empresa (yfinance) ──────────────────────────────
logger.info('Coletando dados de mercado via yfinance...')

def buscar_mercado_anual(ticker, ano_inicio=ANO_INICIO_MACRO, ano_fim=ANO_FIM_MACRO):
    """
    Coleta dados de mercado ano a ano (uma requisição por ano).
    - Mesma estratégia do BCB: evita payload excessivo e delimita o período
    - Cast explícito para datetime64[us] garante compatibilidade com df_key
      no merge_asof (pandas 3.0.1 retorna [s], 3.0.2 retorna [us])
    - Retorna DataFrame com colunas [DT_MKT, retorno_12m, volatilidade_60d]
      ou DataFrame vazio se ticker indisponível/delistado
    """
    try:
        import yfinance as yf
        fragmentos = []
        for ano in range(ano_inicio, ano_fim + 1):
            ini = f'{ano}-01-01'
            fim = f'{ano}-12-31'
            hist = yf.download(ticker, start=ini, end=fim,
                               progress=False, auto_adjust=True)
            if hist.empty:
                continue
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.droplevel(1)
            fragmentos.append(hist[['Close']].copy())

        if not fragmentos:
            return pd.DataFrame()

        close = pd.concat(fragmentos)['Close'].squeeze().sort_index()
        # Normalizar tz e resolução temporal de uma vez
        if close.index.tz is not None:
            close.index = close.index.tz_convert(None)
        # Cast para datetime64[us] — compatibilidade com _dt_naive do dataset
        close.index = close.index.astype('datetime64[us]')

        ret_12m = close.pct_change(252).rename('retorno_12m')
        vol_60d = close.pct_change().rolling(60).std().rename('volatilidade_60d')
        df = pd.concat([ret_12m, vol_60d], axis=1).reset_index()
        df = df.rename(columns={df.columns[0]: 'DT_MKT'})
        return df

    except Exception as e:
        logger.warning('yfinance %s indisponivel: %s', ticker, e)
        return pd.DataFrame()


if 'NOME_CIA' in dataset.columns:
    df_mercado_list = []
    for empresa, ticker in TICKERS_B3.items():
        df_mkt = buscar_mercado_anual(ticker)
        if df_mkt.empty:
            logger.warning('  Mercado %s (%s): sem dados', empresa, ticker)
            continue
        df_mkt['NOME_CIA'] = empresa
        df_mercado_list.append(df_mkt)
        logger.debug('  Mercado %s: %d obs', empresa, len(df_mkt))

    if df_mercado_list:
        df_mercado_all = pd.concat(df_mercado_list, ignore_index=True).sort_values('DT_MKT')

        partes_mkt = []
        for empresa, grp in dataset.sort_values('DT_REFER').groupby('NOME_CIA'):
            df_emp_mkt = df_mercado_all[df_mercado_all['NOME_CIA'] == empresa]
            if df_emp_mkt.empty:
                partes_mkt.append(grp)
                continue

            grp_c = grp.copy()
            # _dt_naive: tz_localize(None) remove tz preservando hora local (00:00)
            # cast para [us] garante compatibilidade com DT_MKT (também [us])
            grp_c['_dt_naive'] = (grp_c['DT_REFER']
                                  .dt.tz_localize(None)
                                  .astype('datetime64[us]'))

            merged = pd.merge_asof(
                grp_c.sort_values('_dt_naive'),
                df_emp_mkt[['DT_MKT', 'retorno_12m', 'volatilidade_60d']],
                left_on='_dt_naive', right_on='DT_MKT',
                direction='backward',
                tolerance=pd.Timedelta('31 days'),
            ).drop(columns=['DT_MKT', '_dt_naive'], errors='ignore')
            partes_mkt.append(merged)

        dataset = pd.concat(partes_mkt, ignore_index=True)
        dataset = dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).reset_index(drop=True)

        cols_mkt = [c for c in dataset.columns if c in ['retorno_12m', 'volatilidade_60d']]
        for c in cols_mkt:
            if c not in cols_macro:
                cols_macro.append(c)
        logger.info('Dados de mercado integrados: %s | cobertura: %s',
                    cols_mkt,
                    {c: f"{dataset[c].notna().mean():.0%}" for c in cols_mkt})
        print(f'Dados de mercado integrados: {cols_mkt}')
    else:
        logger.warning('Dados de mercado indisponiveis — pipeline continua sem eles')
else:
    logger.warning('NOME_CIA nao encontrado — dados de mercado ignorados')

COLS_MACRO_FINAL = [c for c in dataset.columns
                    if c.startswith('macro_') or c in ['retorno_12m', 'volatilidade_60d']]
logger.info('Total colunas macro+mercado: %d', len(COLS_MACRO_FINAL))
print(f'Total colunas macro+mercado adicionadas: {len(COLS_MACRO_FINAL)}')


2026-04-29 21:51:24 | INFO     | Coletando dados de mercado via yfinance...
$RAIZ4.SA: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-12-31) (Yahoo error = "Data doesn't exist for startDate = 1420077600, endDate = 1451527200")

1 Failed download:
['RAIZ4.SA']: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-12-31) (Yahoo error = "Data doesn't exist for startDate = 1420077600, endDate = 1451527200")
$RAIZ4.SA: possibly delisted; no price data found  (1d 2016-01-01 -> 2016-12-31) (Yahoo error = "Data doesn't exist for startDate = 1451613600, endDate = 1483149600")

1 Failed download:
['RAIZ4.SA']: possibly delisted; no price data found  (1d 2016-01-01 -> 2016-12-31) (Yahoo error = "Data doesn't exist for startDate = 1451613600, endDate = 1483149600")
$RAIZ4.SA: possibly delisted; no price data found  (1d 2017-01-01 -> 2017-12-31) (Yahoo error = "Data doesn't exist for startDate = 1483236000, endDate = 1514685600")

1 Failed download:
['RAIZ4.SA']: possibly 

Dados de mercado integrados: ['retorno_12m', 'volatilidade_60d']
Total colunas macro+mercado adicionadas: 7


## Etapa 2 — Deduplicação intra-período

In [6]:
n_antes = len(dataset)
dataset['_n_kpis'] = dataset[kpis_presentes].notna().sum(axis=1)
dataset = (dataset
    .sort_values(['CNPJ_CIA','DT_REFER','ORIGEM','_n_kpis'], ascending=[True,True,True,False])
    .drop_duplicates(subset=['CNPJ_CIA','DT_REFER','ORIGEM'], keep='first')
    .drop(columns=['_n_kpis'])
    .sort_values(['CNPJ_CIA','DT_REFER'])
    .reset_index(drop=True)
)
rem = n_antes - len(dataset)
logger.info("Dedup: %d → %d (-%d)", n_antes, len(dataset), rem)
print(f"Dedup: {n_antes} → {len(dataset)} (removidas {rem} retificações)")


2026-04-29 21:55:23 | INFO     | Dedup: 983 → 983 (-0)


Dedup: 983 → 983 (removidas 0 retificações)


## Etapa 3 — Remoção de colunas com >80% de nulos

In [7]:
COLS_MACRO_FINAL = [c for c in dataset.columns
                    if c.startswith('macro_') or c in ['retorno_12m','volatilidade_60d']]
COLS_PROTEGIDAS = set(kpis_presentes + ['ANO','TRIMESTRE','MES'] + COLS_MACRO_FINAL)
cols_num        = dataset.select_dtypes(include='number').columns.tolist()
cols_cand       = [c for c in cols_num if c not in COLS_PROTEGIDAS]
taxa_nulo       = dataset[cols_cand].isnull().mean()
cols_excluir    = taxa_nulo[taxa_nulo > LIMIAR_NULO].index.tolist()

grupos_exc = {}
for c in cols_excluir:
    p = c.split('_')[0]; grupos_exc[p] = grupos_exc.get(p, 0) + 1
logger.info("Remoção >%.0f%% nulos: %d colunas | %s", LIMIAR_NULO*100, len(cols_excluir),
            dict(sorted(grupos_exc.items(), key=lambda x: -x[1])))

dataset = dataset.drop(columns=cols_excluir)
kpis_presentes = [k for k in kpis_presentes if k in dataset.columns]
print(f"Removidas: {len(cols_excluir)} colunas | Dataset: {dataset.shape} | KPIs: {len(kpis_presentes)}")


2026-04-29 21:55:30 | INFO     | Remoção >80% nulos: 326 colunas | {'BPP': 88, 'BPA': 77, 'DRE': 44, 'DVA': 37, 'DMPL': 35, 'DFC': 34, 'DRA': 11}


Removidas: 326 colunas | Dataset: (983, 452) | KPIs: 19


## Etapa 4 — Imputação por mediana do setor

In [8]:
n_nulos_pre = dataset[kpis_presentes].isnull().sum().sum()

for kpi in kpis_presentes:
    if dataset[kpi].isnull().sum() == 0:
        continue
    m1 = dataset.groupby(['SETOR','ORIGEM'])[kpi].transform('median')
    m2 = dataset.groupby('SETOR')[kpi].transform('median')
    m3 = dataset[kpi].median()
    dataset[kpi] = dataset[kpi].fillna(m1).fillna(m2).fillna(m3)

n_nulos_pos = dataset[kpis_presentes].isnull().sum().sum()
logger.info("Imputação: %d → %d nulos", n_nulos_pre, n_nulos_pos)
print(f"Nulos KPIs: {n_nulos_pre} → {n_nulos_pos}")


2026-04-29 21:56:29 | INFO     | Imputação: 116 → 0 nulos


Nulos KPIs: 116 → 0


## Etapa 5 — Winsorização por setor

Usa `groupby().transform()` — imune ao drop de `SETOR` do pandas 3.x.

In [9]:
def winsorizacao_setor(df_in, col, fator=3.0):
    def _clip(serie):
        q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
        iqr = q3 - q1
        return serie if iqr == 0 else serie.clip(lower=q1-fator*iqr, upper=q3+fator*iqr)
    df_out = df_in.copy()
    df_out[col] = df_in.groupby('SETOR')[col].transform(_clip)
    return df_out

n_clip_total = 0
for kpi in kpis_presentes:
    antes = dataset[kpi].copy()
    dataset = winsorizacao_setor(dataset, kpi, FATOR_WINSOR)
    n_clip_total += (dataset[kpi] != antes).sum()

assert 'SETOR' in dataset.columns, "SETOR perdido na winsorização"
logger.info("Winsorização: fator=%.1f | %d valores truncados", FATOR_WINSOR, n_clip_total)
print(f"✅ Winsorização | fator={FATOR_WINSOR} | {n_clip_total} valores truncados | SETOR: ✅")


2026-04-29 21:56:37 | INFO     | Winsorização: fator=3.0 | 542 valores truncados


✅ Winsorização | fator=3.0 | 542 valores truncados | SETOR: ✅


## Etapa 6 — YoY mesmo trimestre ano anterior (shift=4)

In [10]:
dataset = dataset.sort_values(['CNPJ_CIA','DT_REFER']).reset_index(drop=True)

KPIS_YOY = [k for k in kpis_presentes
             if k not in ('div_liquida','EBITDA','FCF')][:MAX_COLS_YOY]

dataset['_dt_prev'] = dataset.groupby('CNPJ_CIA')['DT_REFER'].shift(4)
dataset['_gap_dias'] = (dataset['DT_REFER'] - dataset['_dt_prev']).dt.days
gap_invalido = ~dataset['_gap_dias'].between(GAP_YOY_MIN, GAP_YOY_MAX)
logger.info("YoY gaps inválidos: %d / %d (%.1f%%)",
            gap_invalido.sum(), len(dataset), gap_invalido.sum()/len(dataset)*100)

cols_yoy = []
for kpi in KPIS_YOY:
    col_yoy = f'{kpi}_yoy'
    prev    = dataset.groupby('CNPJ_CIA')[kpi].shift(4)
    yoy_raw = ((dataset[kpi] - prev) / prev.abs().replace(0, np.nan))
    yoy_raw = yoy_raw.replace([np.inf,-np.inf], np.nan).clip(-CLIP_YOY, CLIP_YOY)
    yoy_raw[gap_invalido] = np.nan
    dataset[col_yoy] = yoy_raw
    cols_yoy.append(col_yoy)

if 'margem_ebitda_yoy' in dataset.columns:
    accel = dataset.groupby('CNPJ_CIA')['margem_ebitda_yoy'].diff()
    accel[gap_invalido] = np.nan
    dataset['aceleracao_ebitda'] = accel
    cols_yoy.append('aceleracao_ebitda')

dataset['pos_ciclo'] = dataset['TRIMESTRE'].astype(float)
dataset.loc[dataset['ORIGEM']=='DFP', 'pos_ciclo'] = 4.0
cols_yoy.append('pos_ciclo')

dataset = dataset.drop(columns=['_dt_prev','_gap_dias'])

logger.info("YoY: %d features | nulos médios: %.0f%%",
            len(cols_yoy), dataset[cols_yoy].isnull().mean().mean()*100)
print(f"Features YoY: {len(cols_yoy)} | nulos médios: {dataset[cols_yoy].isnull().mean().mean():.0%}")


2026-04-29 21:56:44 | INFO     | YoY gaps inválidos: 107 / 983 (10.9%)
2026-04-29 21:56:44 | INFO     | YoY: 18 features | nulos médios: 10%


Features YoY: 18 | nulos médios: 10%


## Etapa 7 — One-hot do setor e flag de origem

In [11]:
dataset = pd.get_dummies(dataset, columns=['SETOR'], prefix='setor', dtype=float)
cols_setor = sorted([c for c in dataset.columns if c.startswith('setor_')])
dataset['flag_dfp'] = (dataset['ORIGEM'] == 'DFP').astype(float)

logger.info("One-hot SETOR: %d colunas | flag_dfp criada", len(cols_setor))
print(f"Setores: {cols_setor}")
print(f"Dataset: {dataset.shape}")


2026-04-29 21:56:50 | INFO     | One-hot SETOR: 5 colunas | flag_dfp criada


Setores: ['setor_Commodities', 'setor_Energia', 'setor_Petróleo', 'setor_Tecnologia', 'setor_Varejo']
Dataset: (983, 475)


## Etapa 8 — Targets: próximo DFP estritamente posterior

Guard para `TZ_DATASET=None`: evita `TypeError: Invalid datetime unit in metadata string "[us, None]"`.

In [12]:
dfp_targets = (
    dataset[dataset['ORIGEM'] == 'DFP']
    [['CNPJ_CIA','DT_REFER'] + list(TARGET_COLS.values())]
    .rename(columns={'DT_REFER': 'DT_DFP',
                     **{v: k for k, v in TARGET_COLS.items()}})
    .sort_values(['CNPJ_CIA','DT_DFP'])
    .reset_index(drop=True)
)

partes = []
for cnpj, grupo in dataset.sort_values(['CNPJ_CIA','DT_REFER']).groupby('CNPJ_CIA'):
    dfp_emp = dfp_targets[dfp_targets['CNPJ_CIA'] == cnpj].sort_values('DT_DFP').reset_index(drop=True)
    g = grupo.sort_values('DT_REFER').copy()

    for tgt_col in TARGET_COLS:
        g[tgt_col] = np.nan
    dt_vals = {}

    for idx in g.index:
        dt_linha = g.loc[idx, 'DT_REFER']
        proximos = dfp_emp[dfp_emp['DT_DFP'] > dt_linha]
        if len(proximos):
            p = proximos.iloc[0]
            for tgt_col in TARGET_COLS:
                g.loc[idx, tgt_col] = p[tgt_col]
            dt_vals[idx] = p['DT_DFP']

    # Guard TZ_DATASET=None
    if dt_vals:
        if TZ_DATASET is not None:
            g['DT_TARGET'] = pd.Series(dt_vals, dtype=f'datetime64[us, {TZ_DATASET}]')
        else:
            g['DT_TARGET'] = pd.Series(dt_vals).astype('datetime64[us]')
    else:
        g['DT_TARGET'] = pd.NaT
    partes.append(g)

dataset = pd.concat(partes, ignore_index=True)
targets_criados = [t for t in TARGET_COLS if t in dataset.columns]

for tgt in targets_criados:
    n_val = dataset[tgt].notna().sum()
    logger.info("Target %-22s: %d/%d válidos (%.0f%%)",
                tgt, n_val, len(dataset), n_val/len(dataset)*100)

print("Targets criados:")
for orig in ['DFP','ITR']:
    sub = dataset[dataset['ORIGEM']==orig]
    val = sub[targets_criados[0]].notna().sum()
    print(f"  {orig}: {val}/{len(sub)} obs com target")

n_ant = len(dataset)
dataset = dataset[dataset[targets_criados].notna().any(axis=1)].reset_index(drop=True)
logger.info("Remoção sem target: %d → %d", n_ant, len(dataset))
print(f"\nDataset com targets: {dataset.shape[0]} × {dataset.shape[1]}")


2026-04-29 21:56:56 | INFO     | Target TARGET_DRE_3.01       : 881/983 válidos (90%)
2026-04-29 21:56:56 | INFO     | Target TARGET_DRE_3.11       : 881/983 válidos (90%)
2026-04-29 21:56:56 | INFO     | Target TARGET_EBITDA         : 881/983 válidos (90%)
2026-04-29 21:56:56 | INFO     | Remoção sem target: 983 → 881


Targets criados:
  DFP: 205/230 obs com target
  ITR: 676/753 obs com target

Dataset com targets: 881 × 479


## Etapa 9 — Seleção de features por correlação e RFE

In [13]:
# dict.fromkeys() preserva ordem e elimina duplicatas (ex: aceleracao_ebitda)
FEATURES_CANDIDATAS = list(dict.fromkeys(
    kpis_presentes
    + cols_yoy       # inclui aceleracao_ebitda e pos_ciclo quando existem
    + cols_setor
    + ['flag_dfp']
    + COLS_MACRO_FINAL
))
FEATURES_CANDIDATAS = [f for f in FEATURES_CANDIDATAS if f in dataset.columns]
logger.info("Features candidatas: %d", len(FEATURES_CANDIDATAS))

target_principal = 'TARGET_DRE_3.01'

if target_principal not in dataset.columns:
    logger.error("Target principal não encontrado")
    FEATURES_SELECIONADAS = FEATURES_CANDIDATAS[:N_FEATURES_RFE]
else:
    df_sel = dataset[FEATURES_CANDIDATAS + [target_principal]].dropna()
    X_sel  = df_sel[FEATURES_CANDIDATAS]
    y_sel  = df_sel[target_principal]

    corr_abs = X_sel.corrwith(y_sel).abs().sort_values(ascending=False)
    FEATURES_CORR = corr_abs[corr_abs >= CORR_MIN].index.tolist()
    logger.info("Pearson: %d → %d features", len(FEATURES_CANDIDATAS), len(FEATURES_CORR))

    n_rfe = min(N_FEATURES_RFE, len(FEATURES_CORR))
    if len(FEATURES_CORR) <= n_rfe:
        FEATURES_SELECIONADAS = FEATURES_CORR
    else:
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(df_sel[FEATURES_CORR])
        rfe = RFE(Ridge(alpha=1.0), n_features_to_select=n_rfe, step=2)
        rfe.fit(X_scaled, y_sel)
        FEATURES_SELECIONADAS = [FEATURES_CORR[i] for i, sel in enumerate(rfe.support_) if sel]
        logger.info("RFE: %d → %d features", len(FEATURES_CORR), len(FEATURES_SELECIONADAS))

    print(f"\nFeatures selecionadas ({len(FEATURES_SELECIONADAS)}):")
    for f in FEATURES_SELECIONADAS:
        print(f"  {f:<35} |r| = {corr_abs.get(f, 0):.3f}")


2026-04-29 21:57:02 | INFO     | Features candidatas: 50
2026-04-29 21:57:02 | INFO     | Pearson: 50 → 21 features
2026-04-29 21:57:02 | INFO     | RFE: 21 → 15 features



Features selecionadas (15):
  EBITDA                              |r| = 0.823
  div_liquida                         |r| = 0.822
  FCF                                 |r| = 0.753
  setor_Petróleo                      |r| = 0.531
  liquidez_corrente                   |r| = 0.216
  setor_Varejo                        |r| = 0.193
  conversao_caixa                     |r| = 0.148
  fco_receita                         |r| = 0.146
  macro_pib_tri                       |r| = 0.137
  giro_ativo_yoy                      |r| = 0.136
  roa_yoy                             |r| = 0.135
  roe_yoy                             |r| = 0.133
  margem_liquida_yoy                  |r| = 0.127
  giro_ativo                          |r| = 0.116
  alavancagem_de_yoy                  |r| = 0.103


## Etapa 10 — Split temporal treino/teste

In [14]:
def calcular_split(grupo, frac=0.75):
    """Retorna Series com labels treino/teste — evita drop de CNPJ_CIA no pandas 3.x."""
    grupo_ord = grupo.sort_values('DT_REFER')
    n    = len(grupo_ord)
    n_tr = max(1, round(n * frac))
    n_tr = min(n_tr, n - 1)
    resultado = pd.Series('teste', index=grupo_ord.index)
    resultado.iloc[:n_tr] = 'treino'
    return resultado

dataset['split'] = dataset.groupby('CNPJ_CIA', group_keys=False).apply(calcular_split)
assert 'CNPJ_CIA' in dataset.columns, "CNPJ_CIA perdido — invariante violado"

treino = dataset[dataset['split'] == 'treino'].copy()
teste  = dataset[dataset['split'] == 'teste'].copy()

contaminados = []
for cnpj, grp in dataset.groupby('CNPJ_CIA'):
    tr_max = grp[grp['split']=='treino']['DT_REFER'].max()
    te_min = grp[grp['split']=='teste']['DT_REFER'].min()
    if pd.notna(tr_max) and pd.notna(te_min) and tr_max > te_min:
        contaminados.append(cnpj)
if contaminados:
    logger.error("Contaminação temporal detectada: %s", contaminados)
else:
    logger.info("Anti-contaminação temporal: PASSOU ✅")

n_tot = len(dataset)
logger.info("Split: %d treino (%.0f%%) | %d teste (%.0f%%)",
            len(treino), len(treino)/n_tot*100, len(teste), len(teste)/n_tot*100)

print(f"Treino : {len(treino)} obs ({len(treino)/n_tot:.0%}) | DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"Teste  : {len(teste)} obs ({len(teste)/n_tot:.0%}) | DFP={(teste['ORIGEM']=='DFP').sum()} | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"CNPJ_CIA preservado: ✅ | Anti-contaminação: {'✅' if not contaminados else '❌'}")


2026-04-29 21:57:06 | INFO     | Anti-contaminação temporal: PASSOU ✅
2026-04-29 21:57:06 | INFO     | Split: 656 treino (74%) | 225 teste (26%)


Treino : 656 obs (74%) | DFP=160 | ITR=496
Teste  : 225 obs (26%) | DFP=45 | ITR=180
CNPJ_CIA preservado: ✅ | Anti-contaminação: ✅


## Etapa 11 — Persistência dos artefatos para o Script 3

In [15]:
TARGETS_VALIDOS  = [t for t in TARGET_COLS if t in dataset.columns]
grupos_treino    = treino['CNPJ_CIA'].values

artefatos_df = {'dataset_preparado': dataset, 'treino': treino, 'teste': teste}
artefatos_meta = {
    'features'      : FEATURES_SELECIONADAS,
    'targets'       : TARGETS_VALIDOS,
    'kpis'          : kpis_presentes,
    'cols_setor'    : cols_setor,
    'cols_yoy'      : cols_yoy,
    'grupos_treino' : grupos_treino,
    'params': {
        'versao'              : 'V6_DFP_ITR',
        'limiar_nulo'         : LIMIAR_NULO,
        'fator_winsor'        : FATOR_WINSOR,
        'max_cols_yoy'        : MAX_COLS_YOY,
        'clip_yoy'            : CLIP_YOY,
        'corr_min'            : CORR_MIN,
        'n_features_rfe'      : N_FEATURES_RFE,
        'frac_treino'         : FRAC_TREINO,
        'gap_yoy_min'         : GAP_YOY_MIN,
        'gap_yoy_max'         : GAP_YOY_MAX,
        'estrategia_target'   : 'proximo_dfp_estritamente_posterior',
        'pandas_version'      : pd.__version__,
        'macro_ano_inicio'    : ANO_INICIO_MACRO,
        'macro_ano_fim'       : ANO_FIM_MACRO,
    },
}

for nome, df_art in artefatos_df.items():
    cam = PASTA_SAIDA / f'{nome}.parquet'
    df_art.to_parquet(cam, index=False)
    logger.info("Salvo: %s | %d × %d | %.0f KB", cam.name, *df_art.shape, cam.stat().st_size/1024)

for nome, obj in artefatos_meta.items():
    cam = PASTA_SAIDA / f'{nome}.pkl'
    with open(cam,'wb') as f: pickle.dump(obj, f)
    logger.info("Salvo: %s", cam.name)

relatorio = {
    'versao'                   : 'V6_DFP_ITR',
    'estrategia_target'        : 'proximo_dfp_estritamente_posterior',
    'dataset_empresas'         : int(dataset['CNPJ_CIA'].nunique()),
    'dataset_anos'             : sorted(dataset['ANO'].dropna().astype(int).unique().tolist()),
    'n_obs_dfp'                : int((dataset['ORIGEM']=='DFP').sum()),
    'n_obs_itr'                : int((dataset['ORIGEM']=='ITR').sum()),
    'n_obs_total'              : int(len(dataset)),
    'n_obs_treino'             : int(len(treino)),
    'n_obs_treino_dfp'         : int((treino['ORIGEM']=='DFP').sum()),
    'n_obs_treino_itr'         : int((treino['ORIGEM']=='ITR').sum()),
    'n_obs_teste'              : int(len(teste)),
    'n_features_candidatas'    : int(len(FEATURES_CANDIDATAS)),
    'n_features_selecionadas'  : int(len(FEATURES_SELECIONADAS)),
    'targets'                  : TARGETS_VALIDOS,
    'features'                 : FEATURES_SELECIONADAS,
    'params'                   : artefatos_meta['params'],
    'nota_groupkfold'          : 'Usar GroupKFold(groups=grupos_treino) no Script 3.',
    'bugs_corrigidos'          : [
        'CAUSA RAIZ cobertura 0%: pd.DataFrame(series_bcb) misturava BCB mensal com EWZ '
            'diário — índice dominado por datas diárias deixava ipca/pib como NaN no '
            'merge_asof. Corrigido com merge_asof individual por série.',
        'BCB coleta ano a ano (loop 2015–2025): evita 406/413 por payload e delimita '
            'o intervalo ao período do TCC.',
        'EWZ tz_convert(None): índice UTC normalizado antes de entrar no dict.',
        'TZ_DATASET=None guard na Etapa 8: evita TypeError com dtype string inválido.',
        'aceleracao_ebitda duplicada: removido ternário explícito; dict.fromkeys() garante unicidade.',
        'Etapa 5: groupby.transform() em vez de apply() — preserva SETOR no pandas 3.x.',
        'Etapa 10: calcular_split retorna Series — preserva CNPJ_CIA no pandas 3.x.',
    ],
}
cam_rel = PASTA_SAIDA / 'relatorio_preparacao_v6.json'
with open(cam_rel,'w',encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*65)
print("  RESUMO FINAL — Script 2 V6 (DFP + ITR)")
print("═"*65)
print(f"  Total obs     : {len(dataset)} (DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()})")
print(f"  Empresas      : {dataset['CNPJ_CIA'].nunique()} / 25")
print(f"  Treino        : {len(treino)} obs ({len(treino)/len(dataset):.0%})")
print(f"    ↳ DFP       : {(treino['ORIGEM']=='DFP').sum()}")
print(f"    ↳ ITR       : {(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste         : {len(teste)} obs ({len(teste)/len(dataset):.0%})")
print(f"  Features      : {len(FEATURES_SELECIONADAS)} selecionadas")
print(f"  Targets       : {TARGETS_VALIDOS}")
print(f"  Vs. V2(DFP)   : {len(dataset)/74:.1f}× mais observações")
print("═"*65)
print("  ⚠️  Script 3: usar GroupKFold(groups=grupos_treino) no CV")
print("  ✅  Pronto para o Script 3 (03_cvm_modelagem.ipynb)")
print("═"*65)


2026-04-29 21:57:13 | INFO     | Salvo: dataset_preparado.parquet | 881 × 480 | 1673 KB
2026-04-29 21:57:13 | INFO     | Salvo: treino.parquet | 656 × 480 | 1303 KB
2026-04-29 21:57:13 | INFO     | Salvo: teste.parquet | 225 × 480 | 664 KB
2026-04-29 21:57:13 | INFO     | Salvo: features.pkl
2026-04-29 21:57:13 | INFO     | Salvo: targets.pkl
2026-04-29 21:57:13 | INFO     | Salvo: kpis.pkl
2026-04-29 21:57:13 | INFO     | Salvo: cols_setor.pkl
2026-04-29 21:57:13 | INFO     | Salvo: cols_yoy.pkl
2026-04-29 21:57:13 | INFO     | Salvo: grupos_treino.pkl
2026-04-29 21:57:13 | INFO     | Salvo: params.pkl



═════════════════════════════════════════════════════════════════
  RESUMO FINAL — Script 2 V6 (DFP + ITR)
═════════════════════════════════════════════════════════════════
  Total obs     : 881 (DFP: 205 | ITR: 676)
  Empresas      : 25 / 25
  Treino        : 656 obs (74%)
    ↳ DFP       : 160
    ↳ ITR       : 496
  Teste         : 225 obs (26%)
  Features      : 15 selecionadas
  Targets       : ['TARGET_DRE_3.01', 'TARGET_DRE_3.11', 'TARGET_EBITDA']
  Vs. V2(DFP)   : 11.9× mais observações
═════════════════════════════════════════════════════════════════
  ⚠️  Script 3: usar GroupKFold(groups=grupos_treino) no CV
  ✅  Pronto para o Script 3 (03_cvm_modelagem.ipynb)
═════════════════════════════════════════════════════════════════
